In [ ]:
# Source - https://stackoverflow.com/a/68614540
# Posted by NL23codes, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-18, License - CC BY-SA 4.0

import pandas
from sklearn.decomposition import PCA
import numpy
import matplotlib.pyplot as plot
from sklearn.preprocessing import StandardScaler

# You must normalize the data before applying the fit method
#df_no_label = df.drop(columns=['species'])
df_no_label = df
scaler = StandardScaler()
X = df_no_label.values.astype(numpy.float16)
X_normalized = scaler.fit_transform(X)  # keep as numpy array, skip the DataFrame wrapper

pca = PCA(n_components=50)
pca.fit(X_normalized)

# Reformat and view results
loadings = pandas.DataFrame(
    pca.components_.T,
    columns=['PC%s' % _ for _ in range(pca.n_components_)],  # use pca.n_components_ to be safe
    index=df_no_label.columns
)
print(loadings)

plot.plot(pca.explained_variance_ratio_)
plot.ylabel('Explained Variance')
plot.xlabel('Components')
plot.show()

In [ ]:
# CODE FOR PCA GENERATED WITH CLAUDE --->
# --- Step 1: Check if 95% variance is already captured ---
cumulative_variance = numpy.cumsum(pca.explained_variance_ratio_)
n_components_90 = numpy.argmax(cumulative_variance >= 0.90) + 1

print(f"Components needed to reach 95% variance: {n_components_90}")
print(f"Variance explained by your current {pca.n_components_} components: {cumulative_variance[-1]:.4f}")

# --- Step 2: Plot cumulative variance with the 90% threshold marked ---
plot.figure(figsize=(10, 5))

plot.subplot(1, 2, 1)
plot.plot(pca.explained_variance_ratio_, marker='o', markersize=3)
plot.ylabel('Explained Variance (per component)')
plot.xlabel('Component index')
plot.title('Per-component variance')

plot.subplot(1, 2, 2)
plot.plot(cumulative_variance, marker='o', markersize=3)
plot.axhline(y=0.90, color='red', linestyle='--', label='90% threshold')
plot.axvline(x=n_components_90 - 1, color='orange', linestyle='--', label=f'{n_components_90} components')
plot.ylabel('Cumulative Explained Variance')
plot.xlabel('Number of components')
plot.title('Cumulative variance')
plot.legend()

plot.tight_layout()
plot.show()

# --- Step 3: Refit with only the components needed for 95% variance ---
pca_reduced = PCA(n_components=n_components_90)
X_reduced = pca_reduced.fit_transform(X_normalized)

print(f"\nOriginal feature count: {X.shape[1]}")
print(f"Reduced component count (95% variance): {X_reduced.shape[1]}")
print(f"Actual variance captured: {sum(pca_reduced.explained_variance_ratio_):.4f}")

In [ ]:
# --- Reduce to 45 components using the already-fitted PCA ---
X_45 = X_normalized @ pca.components_[:45].T
# (equivalent to pca.transform(X_normalized)[:, :45])

print(f"Reduced shape: {X_45.shape}")  # (n_samples, 45)

In [ ]:
import matplotlib.pyplot as plot
import numpy

# Convert one-hot back to integer class labels
species_labels = numpy.argmax(one_hot_df.values, axis=1)  # shape (45337,)

# Optional: get species names if one_hot_df has string column names
species_names = list(one_hot_df.columns)  # e.g. ['cat', 'dog', ...]

# Plot PC1 vs PC2 coloured by species
fig, ax = plot.subplots(figsize=(9, 7))

for i, name in enumerate(species_names):
    mask = species_labels == i
    ax.scatter(
        X_45[mask, 0],
        X_45[mask, 1],
        label=f'{name}',
        alpha=0.4,
        s=10
    )

var = pca.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}%)')
ax.set_title(
    f'PC1 vs PC2  —  {(var[0]+var[1])*100:.1f}% of total variance'
)
ax.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plot.tight_layout()
plot.show()